# TirraMind — HetTGN GNN Retrain on Kaggle or CPU

This notebook resumes training from the **latest epoch checkpoint** found in the dataset and runs up to `--epochs 40` total epochs.

**Current state:** 20 epochs complete. Next run targets epochs 21–40 (Phase 42: enriched entity graph — 19 cftc_tracks + 119 produced_in links).

## Before Running — Two Things to Do

### Step 1 — Make sure two datasets are attached

**Dataset 1: `tirramind-data`**
It must contain a directory with:
- `pipeline.db`  (enriched: 19 cftc_tracks + 119 produced_in links)
- `checkpoints/epoch_020.pt`  (latest checkpoint)

**Dataset 2: `tirramind-code`** (fallback only — not needed if `GITHUB_TOKEN` secret is set)

The notebook does **not** read zip files at runtime. It searches the mounted Kaggle dataset tree for those extracted directories/files directly.

### Step 2 — Attach datasets to THIS notebook

In the Kaggle notebook editor, right panel → **Data** → **Add Dataset**:
- Search `tirramind-data` → Add

Optional but recommended: set **Accelerator → GPU T4 x1** in Settings.

### Step 3 — GitHub Token Secret (for always-latest code)

Add a Kaggle Secret named `GITHUB_TOKEN` with a fine-grained PAT with **read** access to `savabs/tirramind`.
Kaggle → Settings → Secrets → Add New Secret

---
After training, download `gnn_model.pt` and the new epoch checkpoints from the **Output** tab.


## 1. Install and Import Required Libraries

Install `torch-geometric` (and its sparse/scatter kernels) matching the Kaggle-provided PyTorch version.



In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

# torch-geometric — core dependency not bundled in Kaggle's default image
pip("torch-geometric==2.7.0")

# scatter/sparse kernels — pick wheels matching the active torch + device runtime
import torch

torch_ver = torch.__version__.split("+")[0]
cuda_runtime = torch.version.cuda
cuda_tag = f"cu{cuda_runtime.replace('.', '')}" if cuda_runtime else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print(f"Installing PyG extras for torch={torch_ver}, runtime={cuda_tag}")
pip("torch-scatter", "torch-sparse", "-f", wheel_url)

# other lightweight deps
pip("tqdm", "rich")

print("All dependencies installed.")


## 2. Setup Working Directory

**Code** is cloned directly from GitHub (always latest — no dataset re-upload needed for code changes).
To enable this, add a Kaggle Secret named `GITHUB_TOKEN` with a fine-grained personal access token
that has **read** access to the `tirramind_v1` repo.

Go to: Kaggle → Settings → Secrets → Add New Secret → Name: `GITHUB_TOKEN`

If no secret is set, falls back to the `tirramind-code` dataset (backward compatible).

**Data** (`pipeline.db` + epoch checkpoints) is still read from the `tirramind-data` dataset —
these are large binary files that change infrequently and are not in git.



In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. CODE: git clone (preferred) or fall back to dataset ───────────────────
GITHUB_REPO = "savabs/tirramind"   # GitHub username/repo

_cloned_from_git = False
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("tirramind_token")
    _repo_url = f"https://{_token}@github.com/{GITHUB_REPO}.git"

    # Fresh clone every run — always gets the latest trainer.py / scripts
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(
        ["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
        check=True,
        capture_output=True,
    )
    print(f"✓ Cloned {GITHUB_REPO} → {WORK_DIR}  (latest commit, no dataset re-upload needed)")
    _cloned_from_git = True

except Exception as _e:
    print(f"Git clone skipped ({_e.__class__.__name__}: {_e})")
    print("  Falling back to tirramind-code dataset — add GITHUB_TOKEN secret to avoid this.")

    def find_code_root(root: str = "/kaggle/input") -> Path | None:
        for dirpath, dirs, _ in os.walk(root):
            if {"agent", "scripts"}.issubset(set(dirs)):
                return Path(dirpath)
        return None

    code_root = find_code_root()
    assert code_root is not None, (
        "No GITHUB_TOKEN secret and no tirramind-code dataset found. "
        "Either add the secret or attach the dataset."
    )
    for name in ("agent", "scripts"):
        dest = WORK_DIR / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(code_root / name, dest)
        print(f"  Copied {name}/ from dataset")

# ── 2. DATA: pipeline.db comes from Kaggle dataset; checkpoints from HF Hub ──
# Pull latest checkpoint from HF Hub (if HF_TOKEN set) — this is the
# zero-human persistence layer so epochs carry over across Kaggle sessions.
_hg_ckpt_pulled = False
try:
    if _os.environ.get("HF_TOKEN"):
        from huggingface_hub import hf_hub_download, list_repo_files
        _HF_REPO = "savabs/tirramind-hg-data"
        _hf_files = list(list_repo_files(_HF_REPO, repo_type="dataset", token=_os.environ["HF_TOKEN"]))
        _hf_ckpts = sorted([f for f in _hf_files if f.startswith("epoch_") and f.endswith(".pt")])
        if _hf_ckpts:
            _latest_hf = _hf_ckpts[-1]
            _ckpt_dest = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "h_g" / _latest_hf
            _ckpt_dest.parent.mkdir(parents=True, exist_ok=True)
            hf_hub_download(
                repo_id=_HF_REPO, filename=_latest_hf, repo_type="dataset",
                token=_os.environ["HF_TOKEN"], local_dir=str(_ckpt_dest.parent),
            )
            print(f"✓ HF Hub: pulled {_latest_hf} → {_ckpt_dest.parent}")
            # Also pull metrics.jsonl + next_config.json if present
            for _sf in ["metrics.jsonl", "next_config.json", "improvement_history.jsonl"]:
                if _sf in _hf_files:
                    _dest_dir = _ckpt_dest.parent if _sf != "improvement_history.jsonl" else WORK_DIR / "knowledge"
                    _dest_dir.mkdir(parents=True, exist_ok=True)
                    hf_hub_download(
                        repo_id=_HF_REPO, filename=_sf, repo_type="dataset",
                        token=_os.environ["HF_TOKEN"], local_dir=str(_dest_dir),
                    )
                    print(f"  ✓ HF Hub: pulled {_sf}")
            _hg_ckpt_pulled = True
        else:
            print("  [INFO] No epoch_*.pt files in HF Hub repo yet — falling back to Kaggle dataset.")
except Exception as _hf_e:
    print(f"  [INFO] HF Hub pull failed ({_hf_e}) — falling back to Kaggle dataset.")

# ── 2. DATA: pipeline.db + checkpoints always come from the dataset ──────────
def find_data_root(root: str = "/kaggle/input") -> Path | None:
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files) and "checkpoints" in set(dirs):
            return Path(dirpath)
    return None

data_root = find_data_root()
assert data_root is not None, (
    "Could not find tirramind-data dataset containing pipeline.db and checkpoints/. "
    "Attach the dataset in the right panel → Data → Add Dataset."
)

pipeline_db = data_root / "pipeline.db"
ckpt_src    = data_root / "checkpoints"
checkpoint_files = sorted(ckpt_src.glob("epoch_*.pt"))
assert checkpoint_files, f"No epoch_*.pt files found in {ckpt_src}"

pipeline_dir = WORK_DIR / ".tirra_pipeline"
ckpt_dir     = pipeline_dir / "checkpoints"
pipeline_dir.mkdir(exist_ok=True)
ckpt_dir.mkdir(exist_ok=True)

shutil.copy2(pipeline_db, pipeline_dir / "pipeline.db")
print(f"✓ pipeline.db  ({pipeline_db.stat().st_size // 1_000_000} MB)")

for ckpt in checkpoint_files:
    shutil.copy2(ckpt, ckpt_dir / ckpt.name)
    print(f"✓ {ckpt.name}  ({ckpt.stat().st_size // 1_000_000} MB)")

# ── 3. Patch pipeline __init__ to avoid eager APScheduler import ─────────────
pipeline_init = WORK_DIR / "agent" / "pipeline" / "__init__.py"
pipeline_init.write_text(
    '"""TirraMind — Pipeline Layer (Deterministic DAG Scheduler)."""\n\n'
    "from agent.pipeline.storage_backend import (\n"
    "    PostgresBackend,\n    SQLiteBackend,\n    StorageBackend,\n)\n"
    "from agent.pipeline.store import PipelineStore\n\n"
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n\n'
    "def __getattr__(name: str):\n"
    '    if name == "PipelineScheduler":\n'
    "        from agent.pipeline.scheduler import PipelineScheduler\n\n"
    "        return PipelineScheduler\n"
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8",
)
print("✓ Patched agent.pipeline.__init__.py for lazy scheduler import")
print("\nSetup complete.")

# ── 4. Pull training state from GitHub training-state branch ─────────────
print("\nChecking for training state from previous session...")
try:
    _fetch = subprocess.run(
        ["git", "fetch", "origin", "training-state"],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    if _fetch.returncode == 0:
        _state_files = [
            ".tirra_pipeline/checkpoints/h_g/metrics.jsonl",
            ".tirra_pipeline/checkpoints/h_g/next_config.json",
            "knowledge/improvement_history.jsonl",
        ]
        _co = subprocess.run(
            ["git", "checkout", "origin/training-state", "--"] + _state_files,
            cwd=str(WORK_DIR), capture_output=True, text=True,
        )
        if _co.returncode == 0:
            print("✓ Training state pulled from training-state branch.")
        else:
            print(f"  State checkout skipped: {_co.stderr.strip()}")
    else:
        print("  No training-state branch yet — starting fresh.")
except Exception as _e:
    print(f"  State pull skipped ({_e.__class__.__name__}: {_e})")

# ── LLM API key for auto_improve.py ──────────────────────────────────────────
import os as _os
try:
    from kaggle_secrets import UserSecretsClient as _USC
    _llm_key = _USC().get_secret("ANTHROPIC_API_KEY")
    if _llm_key:
        _os.environ["ANTHROPIC_API_KEY"] = _llm_key
        print("  [OK] ANTHROPIC_API_KEY loaded from Kaggle secrets")
except Exception as _e:
    print(f"  [WARN] TIRRA_LLM_API_KEY not set ({_e}) — auto_improve will use heuristic fallback")

# ── wandb API key (for live metric streaming) ─────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient as _USC2
    _wb_key = _USC2().get_secret("WANDB_API_KEY")
    if _wb_key:
        _os.environ["WANDB_API_KEY"] = _wb_key
        _os.environ["WANDB_ENTITY"] = "999-sbpatel"
        print("  [OK] WANDB_API_KEY loaded from Kaggle secrets")
except Exception as _e2:
    print(f"  [INFO] WANDB_API_KEY not set ({_e2}) — wandb logging disabled")

# ── HF Hub token (for checkpoint persistence across sessions) ──────────────────
try:
    from kaggle_secrets import UserSecretsClient as _USC3
    _hf_key = _USC3().get_secret("HF_TOKEN")
    if _hf_key:
        _os.environ["HF_TOKEN"] = _hf_key
        print("  [OK] HF_TOKEN loaded from Kaggle secrets")
except Exception as _e3:
    print(f"  [INFO] HF_TOKEN not set ({_e3}) — checkpoints won't auto-push to HF Hub")



## 3. Environment Check — GPU, PyTorch, PyG Versions


In [ ]:
import torch
import torch_geometric

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU VRAM     : {total_mem} GB")
else:
    print("GPU name     : none")
    print("GPU VRAM     : 0 GB")
print(f"PyG          : {torch_geometric.__version__}")
print(f"Device       : {DEVICE}")

if DEVICE == "cpu":
    print("Running in CPU fallback mode. This is supported, but it will be slower.")


## 4. Pre-flight Check — Verify Required Files Exist


In [ ]:
from pathlib import Path
import shutil

WORK_DIR    = Path("/kaggle/working/tirramind_v1")
PIPELINE_DB = WORK_DIR / ".tirra_pipeline" / "pipeline.db"

# H-G: hypothesis-specific checkpoint subfolder
SHARED_CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints"
CKPT_DIR        = SHARED_CKPT_DIR / "h_g"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Seed: find latest epoch checkpoint from the shared (zip) folder
shared_checkpoints = sorted(SHARED_CKPT_DIR.glob("epoch_*.pt"))
assert shared_checkpoints, (
    f"No seed checkpoints in {SHARED_CKPT_DIR}. "
    "Upload the zip containing epoch_0*.pt before running."
)
latest_shared = shared_checkpoints[-1]
seed_epoch    = int(latest_shared.stem.split("_")[1])  # epoch_030 -> 30

# Copy seed into h_g/ if not already there
seed_dest = CKPT_DIR / latest_shared.name
if not seed_dest.exists():
    shutil.copy2(latest_shared, seed_dest)
    print(f"Seeded {latest_shared.name} → checkpoints/h_g/")

resume_epoch = seed_epoch
print(f"Latest checkpoint : {latest_shared.name}  ({latest_shared.stat().st_size // 1_000_000} MB)")
print(f"H-G checkpoint dir: {CKPT_DIR}")

# Verify pipeline DB
assert PIPELINE_DB.exists(), f"pipeline.db not found at {PIPELINE_DB}"
print(f"pipeline.db       : {PIPELINE_DB.stat().st_size // 1_000_000} MB")

# Verify key source files used by retrain_gnn.py and Trainer
for rel_path in [
    "agent/models/gnn/graph_builder.py",
    "agent/models/gnn/het_tgn.py",
    "agent/models/gnn/trainer.py",
    "agent/models/gnn/temporal.py",
    "agent/models/gnn/ewc.py",
    "agent/models/gnn/alignment.py",
    "scripts/retrain_gnn.py",
    "agent/pipeline/store.py",
]:
    p = WORK_DIR / rel_path
    assert p.exists(), f"Missing source file: {rel_path}"
    print(f"  ✓ {rel_path}")

print(f"\nAll checks passed. H-G will resume from epoch {resume_epoch}.")
RESUME_EPOCH = resume_epoch


## 5. Run GNN Training — H-G: CFTC Derived Features

Runs `scripts/add_cftc_derived_features.py` to enrich the DB with derived
positioning features, then trains with the same architecture as H-A.

**Hypothesis H-G**: CFTC mm_net_pct_52w_rank (crowding percentile),
direction_change, and oi_vs_52w_avg provide normalised context the GNN
cannot infer from raw mm_net_pct_oi values alone → higher ICIR on
commodity futures.

> Architecture: `--num-layers 2 --num-heads 2 --hidden-dim 128` (same as H-A)
> Resumes from epoch 30. New data enrichment happens before training.


In [ ]:
import subprocess
import sys
from pathlib import Path

WORK_DIR    = Path("/kaggle/working/tirramind_v1")
CKPT_DIR    = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "h_g"
PIPELINE_DB = WORK_DIR / ".tirra_pipeline" / "pipeline.db"
DEVICE      = globals().get("DEVICE", "cpu")

# Check if next_config.json was pulled from training-state branch
next_cfg = CKPT_DIR / "next_config.json"

cmd = [
    sys.executable, "scripts/pipeline_orchestrator.py",
    "--work-dir",           str(WORK_DIR),
    "--checkpoint-dir",     str(CKPT_DIR),
    "--db-path",            str(PIPELINE_DB),
    "--knowledge-dir",      str(WORK_DIR / "knowledge"),
    "--block-size",         "5",
    "--total-budget-hours", "11",
    "--device",             DEVICE,
    "--wandb-project",      "tirramind",
    "--hf-repo",            "savabs/tirramind-hg-data",
]
if next_cfg.exists():
    cmd += ["--config-file", str(next_cfg)]
    print("Found next_config.json — applying to first block.")

print("Running:", " ".join(cmd))
print("-" * 70)

retcode = subprocess.call(cmd, cwd=str(WORK_DIR))
print(f"\nPipeline orchestrator exited: {retcode}")
if retcode in (2, 3):
    print("Human review required — check GitHub Issues.")
elif retcode == 4:
    print("Training crashed — check logs above for OOM or CUDA error.")
elif retcode == 1:
    print("Config updated — next Kaggle session will apply new settings.")
else:
    print("Session complete — next run triggered via GitHub Actions.")


## 6. Verify Outputs and Prepare for Download


In [ ]:
import shutil
from pathlib import Path

WORK_DIR   = Path("/kaggle/working/tirramind_v1")
CKPT_DIR   = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "h_g"
OUT_DIR    = Path("/kaggle/working")

# Epochs already on the laptop before this run — skip re-copying those
ALREADY_HAVE_UP_TO = resume_epoch  # set by Cell 9 above

print("=== H-G checkpoints saved ===")
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    size_mb = ckpt.stat().st_size / 1_000_000
    print(f"  {ckpt.name}  ({size_mb:.1f} MB)")

# Copy final model
final_model = WORK_DIR / ".tirra_pipeline" / "gnn_model_h_g.pt"
if final_model.exists():
    shutil.copy2(final_model, OUT_DIR / "gnn_model_h_g.pt")
    print(f"\ngnn_model_h_g.pt → /kaggle/working/ ({final_model.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("\ngnn_model_h_g.pt not found — training may not have completed yet.")

# Copy only NEW checkpoints (those produced this run)
new_checkpoints = []
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    epoch_num = int(ckpt.stem.split("_")[1])
    if epoch_num > ALREADY_HAVE_UP_TO:
        dest = OUT_DIR / ckpt.name
        shutil.copy2(ckpt, dest)
        new_checkpoints.append(ckpt.name)
        print(f"{ckpt.name} → /kaggle/working/")

print("\n=== Download from the 'Output' tab ===")
print("Files to grab:")
print("  gnn_model_h_g.pt")
for name in new_checkpoints:
    print(f"  {name}")


## 7. Copy H-G Results Back to Laptop

After the Kaggle session finishes, download the output files and run:

```bash
cd /home/becmachlean/2024/projects/tirramind_v1

# Save H-G final model
cp ~/Downloads/gnn_model_h_g.pt .tirra_pipeline/gnn_model_h_g.pt

# Copy new epoch checkpoints into the h_g/ subfolder
mkdir -p .tirra_pipeline/checkpoints/h_g/
for f in ~/Downloads/epoch_*.pt; do
    [ -f "$f" ] && cp "$f" .tirra_pipeline/checkpoints/h_g/
done

# Verify
ls -lh .tirra_pipeline/checkpoints/h_g/
ls -lh .tirra_pipeline/gnn_model_h_g.pt

# Run IC diagnostic backtest
python scripts/phase40_gnn_backtest.py --model .tirra_pipeline/gnn_model_h_g.pt

# Compare H-G vs H-A baseline (did CFTC derived features improve IC?)
python scripts/compare_experiments.py --latest 2
```

**Next run:** update `TARGET_EPOCHS` in Cell 11 to the new target (e.g. 50).
The notebook auto-detects the latest checkpoint in `checkpoints/h_g/` and resumes from there.
